# PerfGazer Quickstart

This notebook demonstrates how to set up and use PerfGazer to capture Spark performance metrics.

**Works on:** Local (Almond/Jupyter) | Databricks

## 1. Setup Dependencies

Skip this cell on Databricks if PerfGazer is installed as a cluster library.

In [1]:
// Local setup only - load dependencies via Ammonite/Almond
import $ivy.`org.apache.spark::spark-sql:3.5.2`
import $ivy.`io.github.amadeusitgroup::perfgazer_spark_3-5-2:0.0.1`

import $ivy.$
import $ivy.$

## 2. Create SparkSession and Register PerfGazer

Create a SparkSession and register PerfGazer programmatically.

In [2]:
import org.apache.spark.sql.SparkSession
import com.amadeus.perfgazer.{JsonSink, PerfGazer}

// Output directory for reports
val outputDir = "/tmp/perfgazer-demo"

val spark = SparkSession.builder()
  .appName("PerfGazer Demo")
  .master("local[*]")  // Remove this line on Databricks
  .config("spark.driver.memory", "2g")  // Remove on Databricks
  .getOrCreate()

// Create and register PerfGazer
val jsonSink = new JsonSink(
  JsonSink.Config(destination = outputDir),
  spark.conf
)
val perfgazer = new PerfGazer(sink = jsonSink)
spark.sparkContext.addSparkListener(perfgazer)

println(s"SparkSession created. Reports will be written to: $outputDir")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 07:07:56 INFO SparkContext: Running Spark version 3.5.2
26/04/02 07:07:56 INFO SparkContext: OS info Linux, 6.11.3-200.fc40.aarch64, amd64
26/04/02 07:07:56 INFO SparkContext: Java version 1.8.0_442
26/04/02 07:07:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/02 07:07:56 INFO ResourceUtils: ==============================================================
26/04/02 07:07:56 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/02 07:07:56 INFO ResourceUtils: ==============================================================
26/04/02 07:07:56 INFO SparkContext: Submitted application: PerfGazer Demo
26/04/02 07:07:56 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor

SparkSession created. Reports will be written to: /tmp/perfgazer-demo


import org.apache.spark.sql.SparkSession
outputDir: String = "/tmp/perfgazer-demo"
spark: SparkSession = org.apache.spark.sql.SparkSession@6df06bf4

## 3. Run Some Spark Operations

PerfGazer automatically captures metrics for all Spark jobs, stages, and SQL queries.

In [3]:
import spark.implicits._

// Create a sample dataset
val data = spark.range(1, 1000000)
  .withColumn("category", ($"id" % 10).cast("string"))
  .withColumn("value", $"id" * 2)

// Trigger some Spark actions
val result = data
  .groupBy("category")
  .agg(
    org.apache.spark.sql.functions.count("*").as("count"),
    org.apache.spark.sql.functions.sum("value").as("total_value"),
    org.apache.spark.sql.functions.avg("value").as("avg_value")
  )
  .orderBy("category")

result.show()

26/04/02 07:08:09 INFO SharedState: Setting hive.metastore.warehouse.dir ('null') to the value of spark.sql.warehouse.dir.
26/04/02 07:08:09 INFO SharedState: Warehouse path is 'file:/home/jovyan/work/notebooks/spark-warehouse'.
26/04/02 07:08:13 INFO CodeGenerator: Code generated in 479.874068 ms
26/04/02 07:08:13 INFO DAGScheduler: Registering RDD 3 (show at cmd3.sc:18) as input to shuffle 0
26/04/02 07:08:13 INFO DAGScheduler: Got map stage job 0 (show at cmd3.sc:18) with 5 output partitions
26/04/02 07:08:13 INFO DAGScheduler: Final stage: ShuffleMapStage 0 (show at cmd3.sc:18)
26/04/02 07:08:13 INFO DAGScheduler: Parents of final stage: List()
26/04/02 07:08:13 INFO DAGScheduler: Missing parents: List()
26/04/02 07:08:13 INFO DAGScheduler: Submitting ShuffleMapStage 0 (MapPartitionsRDD[3] at show at cmd3.sc:18), which has no missing parents
26/04/02 07:08:13 INFO MemoryStore: Block broadcast_0 stored as values in memory (estimated size 45.1 KiB, free 876.0 MiB)
26/04/02 07:08:13 I

+--------+------+------------+---------+
|category| count| total_value|avg_value|
+--------+------+------------+---------+
|       0| 99999| 99999000000|1000000.0|
|       1|100000| 99999200000| 999992.0|
|       2|100000| 99999400000| 999994.0|
|       3|100000| 99999600000| 999996.0|
|       4|100000| 99999800000| 999998.0|
|       5|100000|100000000000|1000000.0|
|       6|100000|100000200000|1000002.0|
|       7|100000|100000400000|1000004.0|
|       8|100000|100000600000|1000006.0|
|       9|100000|100000800000|1000008.0|
+--------+------+------------+---------+



import spark.implicits._
data: org.apache.spark.sql.package.DataFrame = [id: bigint, category: string ... 1 more field]
result: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [category: string, count: bigint ... 2 more fields]

In [4]:
// Another operation - join example
val categories = Seq(
  ("0", "Category A"),
  ("1", "Category B"),
  ("2", "Category C")
).toDF("category", "name")

val joined = result.join(categories, "category")
joined.show()

26/04/02 07:08:21 INFO CodeGenerator: Code generated in 41.32943 ms
26/04/02 07:08:22 INFO CodeGenerator: Code generated in 5.685213 ms
26/04/02 07:08:22 INFO CodeGenerator: Code generated in 9.431709 ms
26/04/02 07:08:22 INFO SparkContext: Starting job: $anonfun$withThreadLocalCaptured$1 at FutureTask.java:266
26/04/02 07:08:22 INFO DAGScheduler: Got job 2 ($anonfun$withThreadLocalCaptured$1 at FutureTask.java:266) with 3 output partitions
26/04/02 07:08:22 INFO DAGScheduler: Final stage: ResultStage 3 ($anonfun$withThreadLocalCaptured$1 at FutureTask.java:266)
26/04/02 07:08:22 INFO DAGScheduler: Parents of final stage: List()
26/04/02 07:08:22 INFO DAGScheduler: Missing parents: List()
26/04/02 07:08:22 INFO DAGScheduler: Submitting ResultStage 3 (MapPartitionsRDD[10] at $anonfun$withThreadLocalCaptured$1 at FutureTask.java:266), which has no missing parents
26/04/02 07:08:22 INFO MemoryStore: Block broadcast_2 stored as values in memory (estimated size 5.2 KiB, free 875.9 MiB)
26/0

+--------+------+-----------+---------+----------+
|category| count|total_value|avg_value|      name|
+--------+------+-----------+---------+----------+
|       0| 99999|99999000000|1000000.0|Category A|
|       1|100000|99999200000| 999992.0|Category B|
|       2|100000|99999400000| 999994.0|Category C|
+--------+------+-----------+---------+----------+



categories: org.apache.spark.sql.package.DataFrame = [category: string, name: string]
joined: org.apache.spark.sql.package.DataFrame = [category: string, count: bigint ... 3 more fields]

## 4. View Generated Reports

PerfGazer generates JSON reports for:
- **SQL reports** - query-level metrics and execution plans
- **Job reports** - job timing and stage information  
- **Stage reports** - detailed stage metrics

In [5]:
// List generated report files
import java.io.File

val reportFiles = new File(outputDir).listFiles()
  .filter(_.getName.endsWith(".json"))
  .map(_.getName)
  .sorted

println("Generated report files:")
reportFiles.foreach(println)

Generated report files:
job-reports-6b9ec440-ca5f-4b84-9bdf-b8863f68751d.json
sql-reports-cfed886b-baec-4d0a-8b1b-f65515ff5b4a.json
stage-reports-d1fb5fde-0beb-46f3-84c0-ae9221787d33.json
task-reports-df6512d3-7832-4dd3-8bb4-5c901d08fe9a.json


import java.io.File
reportFiles: Array[String] = Array(
  "job-reports-6b9ec440-ca5f-4b84-9bdf-b8863f68751d.json",
  "sql-reports-cfed886b-baec-4d0a-8b1b-f65515ff5b4a.json",
  "stage-reports-d1fb5fde-0beb-46f3-84c0-ae9221787d33.json",
  "task-reports-df6512d3-7832-4dd3-8bb4-5c901d08fe9a.json"
)

## 5. Query Reports with Spark SQL

The reports are JSON files that can be queried directly with Spark.

In [6]:
// Create views for the reports
spark.sql(s"""
  CREATE OR REPLACE TEMPORARY VIEW job
  USING json
  OPTIONS (path '$outputDir/job-reports-*.json')
""")

spark.sql(s"""
  CREATE OR REPLACE TEMPORARY VIEW sql
  USING json
  OPTIONS (path '$outputDir/sql-reports-*.json')
""")

spark.sql(s"""
  CREATE OR REPLACE TEMPORARY VIEW stage
  USING json
  OPTIONS (path '$outputDir/stage-reports-*.json')
""")

26/04/02 07:08:27 INFO InMemoryFileIndex: It took 22 ms to list leaf files for 1 paths.
26/04/02 07:08:27 INFO InMemoryFileIndex: It took 5 ms to list leaf files for 1 paths.
26/04/02 07:08:27 INFO FileSourceStrategy: Pushed Filters: 
26/04/02 07:08:27 INFO FileSourceStrategy: Post-Scan Filters: 
26/04/02 07:08:27 INFO MemoryStore: Block broadcast_6 stored as values in memory (estimated size 350.0 KiB, free 867.5 MiB)
26/04/02 07:08:27 INFO MemoryStore: Block broadcast_6_piece0 stored as bytes in memory (estimated size 34.2 KiB, free 867.5 MiB)
26/04/02 07:08:27 INFO BlockManagerInfo: Added broadcast_6_piece0 in memory on 9c740b1901d0:43533 (size: 34.2 KiB, free: 875.9 MiB)
26/04/02 07:08:27 INFO SparkContext: Created broadcast 6 from sql at cmd6.sc:5
26/04/02 07:08:27 INFO FileSourceScanExec: Planning scan with bin packing, max size: 4194304 bytes, open cost is considered as scanning 4194304 bytes.
26/04/02 07:08:27 INFO SparkContext: Starting job: sql at cmd6.sc:5
26/04/02 07:08:27 I

res6_0: org.apache.spark.sql.package.DataFrame = []
res6_1: org.apache.spark.sql.package.DataFrame = []
res6_2: org.apache.spark.sql.package.DataFrame = []

In [7]:
// Query job reports
spark.sql("""
  SELECT 
    *
  FROM job

""").show(truncate = false)

26/04/02 07:08:28 INFO FileSourceStrategy: Pushed Filters: 
26/04/02 07:08:28 INFO FileSourceStrategy: Post-Scan Filters: 
26/04/02 07:08:28 INFO MemoryStore: Block broadcast_9 stored as values in memory (estimated size 349.9 KiB, free 866.4 MiB)
26/04/02 07:08:28 INFO MemoryStore: Block broadcast_9_piece0 stored as bytes in memory (estimated size 34.2 KiB, free 866.4 MiB)
26/04/02 07:08:28 INFO BlockManagerInfo: Added broadcast_9_piece0 in memory on 9c740b1901d0:43533 (size: 34.2 KiB, free: 875.8 MiB)
26/04/02 07:08:28 INFO SparkContext: Created broadcast 9 from show at cmd7.sc:7
26/04/02 07:08:28 INFO FileSourceScanExec: Planning scan with bin packing, max size: 4194304 bytes, open cost is considered as scanning 4194304 bytes.


++
||
++
++



In [8]:
// Query SQL reports
spark.sql("""
  SELECT 
    sqlId,
    description
  FROM sql
  ORDER BY sqlId
""").show(truncate = false)

org.apache.spark.sql.catalyst.ExtendedAnalysisException: [UNRESOLVED_COLUMN.WITHOUT_SUGGESTION] A column or function parameter with name `sqlId` cannot be resolved. ; line 3 pos 4;
'Sort ['sqlId ASC NULLS FIRST], true
+- 'Project ['sqlId, 'description]
   +- SubqueryAlias sql
      +- View (`sql`, [])
         +- Relation [] json


## 6. Cleanup

In [ ]:
spark.stop()

## Configuration Reference

| Property | Default | Description |
|----------|---------|-------------|
| `spark.perfgazer.sink.class` | (required) | Sink class: `JsonSink` or `LogSink` |
| `spark.perfgazer.sink.json.destination` | (required for JsonSink) | Output directory |
| `spark.perfgazer.sql.enabled` | `true` | Enable SQL reports |
| `spark.perfgazer.jobs.enabled` | `true` | Enable job reports |
| `spark.perfgazer.stages.enabled` | `true` | Enable stage reports |
| `spark.perfgazer.tasks.enabled` | `false` | Enable task reports (verbose) |
| `spark.perfgazer.max.cache.size` | `100` | Max events to cache per type |